# Module 04: Feature Engineering

## What You'll Learn

- On-Demand Feature Views (real-time transformations)
- Batch Feature Views (scheduled transformations)
- Push Sources for streaming/real-time data
- Feature Services (grouping features for use cases)
- Transformation patterns and limitations

---

> **🗺️ DATA STRATEGY**: Feature engineering in Feast supports the "prototype-to-production" principle from the strategy. Simple ODFVs for rapid prototyping, graduating to batch/streaming sources as workloads mature.

## On-Demand Feature Views (ODFVs)

ODFVs compute features **at request time** — they transform input data or combine existing features on-the-fly without storing results.

Use cases:
- Derived features (ratios, differences, combinations)
- Features that depend on request-time context
- Rapid prototyping before committing to batch materialization

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

from feast import (
    Entity, FeatureView, Field, FileSource, FeatureStore,
    FeatureService, PushSource, BatchFeatureView
)
from feast.on_demand_feature_view import on_demand_feature_view
from feast.types import Float32, Float64, Int64, String

# Generate base data
np.random.seed(42)
os.makedirs("data", exist_ok=True)

records = []
for customer_id in range(1, 51):
    for week in range(26):
        ts = datetime(2024, 1, 1) + timedelta(weeks=week)
        records.append({
            "customer_id": customer_id,
            "event_timestamp": ts,
            "annual_income": np.random.randint(30000, 200000),
            "total_debt": np.random.randint(0, 100000),
            "monthly_expenses": np.random.randint(1000, 8000),
            "credit_limit": np.random.randint(5000, 50000),
            "current_balance": np.random.randint(0, 40000),
        })

df = pd.DataFrame(records)
df.to_parquet("data/customer_financials.parquet")
print(f"Generated {len(df)} records")

In [ ]:
# Define base feature view
customer = Entity(name="customer", join_keys=["customer_id"])

financials_source = FileSource(
    name="customer_financials_source",
    path=os.path.abspath("data/customer_financials.parquet"),
    timestamp_field="event_timestamp",
)

financials_fv = FeatureView(
    name="customer_financials",
    entities=[customer],
    ttl=timedelta(weeks=2),
    schema=[
        Field(name="annual_income", dtype=Int64),
        Field(name="total_debt", dtype=Int64),
        Field(name="monthly_expenses", dtype=Int64),
        Field(name="credit_limit", dtype=Int64),
        Field(name="current_balance", dtype=Int64),
    ],
    source=financials_source,
)
print("Base feature view defined")

In [ ]:
# On-Demand Feature View: compute derived features at request time
@on_demand_feature_view(
    sources=[financials_fv],
    schema=[
        Field(name="debt_to_income_ratio", dtype=Float64),
        Field(name="credit_utilization", dtype=Float64),
        Field(name="monthly_savings_rate", dtype=Float64),
        Field(name="risk_category", dtype=String),
    ],
)
def financial_ratios(inputs: pd.DataFrame) -> pd.DataFrame:
    """Compute financial risk ratios from base features."""
    result = pd.DataFrame()
    result["debt_to_income_ratio"] = inputs["total_debt"] / inputs["annual_income"].clip(lower=1)
    result["credit_utilization"] = inputs["current_balance"] / inputs["credit_limit"].clip(lower=1)
    result["monthly_savings_rate"] = (
        (inputs["annual_income"] / 12 - inputs["monthly_expenses"]) / (inputs["annual_income"] / 12).clip(lower=1)
    )
    result["risk_category"] = pd.cut(
        result["debt_to_income_ratio"],
        bins=[0, 0.3, 0.5, 0.7, float("inf")],
        labels=["low", "moderate", "high", "very_high"],
    ).astype(str)
    return result

print("On-Demand Feature View 'financial_ratios' defined")
print("  - Computes: debt_to_income_ratio, credit_utilization, monthly_savings_rate, risk_category")
print("  - Computed AT REQUEST TIME (not materialized)")

In [ ]:
# Apply everything
os.makedirs("feature_repo", exist_ok=True)
store = FeatureStore(repo_path="feature_repo")
store.apply([customer, financials_source, financials_fv, financial_ratios])
print("✅ Applied base features + on-demand transformations")

In [ ]:
# Retrieve features including the on-demand computed ones
entity_df = pd.DataFrame({
    "customer_id": [1, 10, 25, 40],
    "event_timestamp": [datetime(2024, 3, 1)] * 4,
})

result = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "customer_financials:annual_income",
        "customer_financials:total_debt",
        "financial_ratios:debt_to_income_ratio",
        "financial_ratios:credit_utilization",
        "financial_ratios:risk_category",
    ],
).to_df()

print("Features with on-demand transformations:")
result

## Feature Services

A **Feature Service** groups features for a specific use case. It's the contract between the feature store and the model — "these are the features this model needs."

In [ ]:
# Define a Feature Service for the credit scoring model
credit_scoring_service = FeatureService(
    name="credit_scoring_v1",
    features=[
        financials_fv[["annual_income", "total_debt", "credit_limit"]],
        financial_ratios,
    ],
    description="All features needed for credit scoring model v1",
)

store.apply([credit_scoring_service])
print("✅ Feature Service 'credit_scoring_v1' registered")
print("\nThis is the contract: when the model asks for features,")
print("it references the Feature Service name, not individual features.")

## Push Sources (Streaming / Real-Time)

**Push Sources** allow you to push feature values directly into the online store without waiting for batch materialization. This enables near-real-time feature updates.

> **⚠️ GAP**: Streaming support is at **Alpha** maturity. Push-only model — no native Flink or Spark Streaming integration. No continuous aggregation. Features can be up to 24 hours stale even with push-based streaming due to materialization scheduling defaults. This is a P3 priority in the data strategy.
>
> **⚠️ GAP**: No tiled streaming aggregation architecture. Chronon (competitor) achieves sub-10ms serving for streaming features via a tiled architecture that reduces reads from O(events) to O(tiles).

In [ ]:
# Define a Push Source for real-time transaction events
transaction_push_source = PushSource(
    name="transaction_events",
    batch_source=FileSource(
        name="transaction_batch_source",
        path=os.path.abspath("data/customer_financials.parquet"),
        timestamp_field="event_timestamp",
    ),
)

# Feature view backed by push source
realtime_fv = FeatureView(
    name="realtime_transactions",
    entities=[customer],
    ttl=timedelta(hours=1),  # Short TTL for real-time features
    schema=[
        Field(name="current_balance", dtype=Int64),
        Field(name="monthly_expenses", dtype=Int64),
    ],
    source=transaction_push_source,
)

store.apply([transaction_push_source, realtime_fv])
print("✅ Push source and real-time feature view defined")
print("\nPush sources allow writing features directly to online store:")
print("  store.push('transaction_events', df)")

In [ ]:
# Push real-time data
realtime_df = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "event_timestamp": [datetime.now()] * 3,
    "current_balance": [15000, 8500, 22000],
    "monthly_expenses": [3200, 2100, 5500],
})

store.push("transaction_events", realtime_df)
print("✅ Pushed real-time transaction data for 3 customers")
print("\nThese values are now immediately available via get_online_features()")
print("No materialization needed for pushed data.")

## Transformation Patterns Summary

| Pattern | When to Use | Computed | Stored |
|---------|-------------|----------|--------|
| **Batch FeatureView** | Historical data, scheduled refresh | At materialization time | Yes (offline + online) |
| **On-Demand FV** | Derived features, request-time context | At request time | No |
| **Push Source** | Real-time events, streaming | At push time | Yes (online only) |

> **⚠️ GAP**: Feast requires **separate definitions** for batch, streaming, and serving features. Unlike Chronon's unified temporal computation model, you cannot define a feature once and have it automatically serve batch, streaming, and real-time patterns. This is a fundamental architectural difference and a long-term gap.

## Key Takeaways

1. **On-Demand Feature Views** compute derived features at request time (ratios, categories, combinations)
2. **Feature Services** group features into contracts for specific models
3. **Push Sources** enable near-real-time feature updates without batch materialization
4. Streaming is Alpha — significant gaps vs competitors in continuous aggregation
5. No unified computation model — batch and streaming require separate definitions

## What's Next

- **Module 05**: Configuration Deep Dive — `feature_store.yaml`, CRD, multi-backend setup
- **Module 06**: Ray Compute — distributed transformations and materialization